# PlantSeg Stage 6 — Revision Analyses (Array review, September 2026)

Four analyses requested (directly or indirectly) by the five *Array* reviewers, all reusing
**existing trained checkpoints** — no retraining. Run all cells top to bottom on Kaggle with
GPU enabled, the PlantSeg dataset attached, and the Stage-1/Stage-2/deployment checkpoints
attached as input datasets (see `CKPT_*` paths in the config cell — edit if your dataset
mount points differ).

1. **Classifier-stage Grad-CAM validated against ground-truth lesion masks** (Reviewers 4 & 5,
   most critical point: the deployed confidence flag uses ConvNeXtV2-Tiny's own Grad-CAM, which
   Section 4 of the paper never actually scored against ground truth — only the *segmenters'*
   Grad-CAM was validated there).
2. **Model/ensemble selection re-run using the validation set instead of the test set**
   (Reviewers 4 & 5: test-set model selection followed by test-set evaluation is optimistic).
3. **Deployed pipeline performance on the "clean" vs. cross-split-URL-leaked test subset**
   (Reviewer 5, point 8: group/session independence of the split).
4. **Ground-truth-independent CAM threshold sensitivity check** for the six segmenters'
   faithfulness scores (Reviewers 3, 4, 5: the original CAM–GT IoU threshold is set from the
   ground-truth lesion area, which is an oracle-assisted evaluation choice).

Each section writes its own `OUT_*.csv` so results can be pulled back into the paper.

In [1]:
!pip install -q timm segmentation-models-pytorch grad-cam transformers
import os, json, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
import timm
import segmentation_models_pytorch as smp
from pytorch_grad_cam import GradCAM
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.1 MB/s eta 0:00:00
Device: cuda


## Config — edit these paths to match your Kaggle input datasets

In [2]:
# ============ CONFIG ============
DATA_ROOT = Path("/kaggle/input/datasets/mdmarufbangabashi/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for cand in [Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): DATA_ROOT = cand; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not found under {DATA_ROOT}"

# Classifier + segmenter checkpoints used by the deployed pipeline
CLS_CKPT = Path("/kaggle/input/datasets/mdmarufbangabashi/stage-1-pt/ConvNeXtV2Tiny_best.pt")
SEG_CKPT_BEST = Path("/kaggle/input/datasets/mdmarufbangabashi/stage-2-pt/DeepLabV3Plus_efficientnet-b3.pt")

# All 8 classifier checkpoints (Stage 1) and all 6 segmenter checkpoints (Stage 2),
# for the validation-based re-ranking in Section 2. Edit the parent dir to your dataset mount.
CLS_CKPT_DIR = Path("/kaggle/input/datasets/mdmarufbangabashi/stage-1-pt")
SEG_CKPT_DIR = Path("/kaggle/input/datasets/mdmarufbangabashi/stage-2-pt")

CLS_CKPTS = {
    "MobileNetV3Large": CLS_CKPT_DIR/"MobileNetV3Large_best.pt",
    "EfficientNetB0":   CLS_CKPT_DIR/"EfficientNetB0_best.pt",
    "ResNet50":          CLS_CKPT_DIR/"ResNet50_best.pt",
    "EfficientNetV2S":   CLS_CKPT_DIR/"EfficientNetV2S_best.pt",
    "ConvNeXtV2Tiny":    CLS_CKPT_DIR/"ConvNeXtV2Tiny_best.pt",
    "ViTBase16":         CLS_CKPT_DIR/"ViTBase16_best.pt",
    "SwinTiny":          CLS_CKPT_DIR/"SwinTiny_best.pt",
    "GatedFusionLight":  CLS_CKPT_DIR/"GatedFusionLight_best.pt",
}
SEG_CKPTS = {
    "UNet_resnet34":                 SEG_CKPT_DIR/"UNet_resnet34.pt",
    "UNetPP_resnet50":               SEG_CKPT_DIR/"UNetPP_resnet50.pt",
    "FPN_resnet50":                  SEG_CKPT_DIR/"FPN_resnet50.pt",
    "DeepLabV3Plus_resnet50":        SEG_CKPT_DIR/"DeepLabV3Plus_resnet50.pt",
    "DeepLabV3Plus_efficientnet-b3": SEG_CKPT_DIR/"DeepLabV3Plus_efficientnet-b3.pt",
    "SegFormer_b2":                  SEG_CKPT_DIR/"SegFormer_b2.pt",
}

OUTDIR = Path("/kaggle/working/stage6_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS = 384      # matches the corrected Methods text: CNN classifiers trained at 384x384
IMG_SIZE_CLS_TF = 224   # ViT-B/16 / Swin-Tiny are fixed at 224x224
IMG_SIZE_SEG = 512      # match Stage-2/Stage-4's segmentation input size -- EDIT if different
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["maskpath"] = meta.apply(lambda r: DATA_ROOT/"annotations"/r["split"]/r["Label file"], axis=1)
meta["label"] = meta["Disease"]
CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)
print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

def load_state(model, ckpt_path):
    raw = torch.load(ckpt_path, map_location="cpu")
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    raw = {k.replace("module.", ""): v for k, v in raw.items()}
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if missing or unexpected:
        print(f"  [load_state:{ckpt_path.name}] missing={len(missing)} unexpected={len(unexpected)}")
    return model

classes=115  train=5367  val=846  test=1561


## Section 1 — Classifier-stage Grad-CAM validated against ground truth

Same methodology as Stage 4 (`cam_gt_metrics` / `deletion_auc`), applied to ConvNeXtV2-Tiny
(the deployed classifier) on the **same 300-image stratified subset** used for the segmenter
faithfulness table, so the numbers are directly comparable to Table 10.

In [3]:
# ============ SECTION 1: CLASSIFIER GRAD-CAM VS GROUND TRUTH ============
NAMES_300 = ['apple_black_rot_google_0037.jpg', 'apple_black_rot_google_0077.jpg', 'apple_mosaic_virus_57.jpg', 'apple_mosaic_virus_google_0074.jpg', 'apple_rust_147.jpg', 'apple_rust_google_0007.jpg', 'apple_rust_google_0011.jpg', 'apple_scab_50.jpg', 'apple_scab_85.jpg', 'apple_scab_google_0018.jpg', 'apple_scab_google_0031.jpg', 'apple_scab_google_0153.jpg', 'apple_scab_google_0244.jpg', 'banana_anthracnose_Baidu_0059.jpg', 'banana_anthracnose_Bing_0006.jpg', 'banana_anthracnose_Bing_0022.jpg', 'banana_anthracnose_Bing_0032.jpg', 'banana_black_leaf_streak_Bing_0124.jpg', 'banana_black_leaf_streak_banana black sigatoka (26).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_bunchy_top_Google_0037.jpg', 'banana_cigar_end_rot_Bing_0007.jpg', 'banana_cigar_end_rot_Google_0101.jpg', 'banana_cordana_leaf_spot_Bing_0007.jpg', 'banana_cordana_leaf_spot_Bing_0037.jpg', 'banana_panama_disease_126.jpg', 'banana_panama_disease_google_0064.jpg', 'basil_downy_mildew_google_0031.jpg', 'basil_downy_mildew_google_0091.jpg', 'bean_halo_blight_11.jpg', 'bean_halo_blight_2.jpg', 'bean_mosaic_virus_24.jpg', 'bean_mosaic_virus_google_0098.jpg', 'bean_rust_google_0045.jpg', 'bean_rust_google_0068.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0028.jpg', 'bell_pepper_blossom_end_rot_Bing_0002.jpg', 'bell_pepper_blossom_end_rot_Bing_0004.jpg', 'bell_pepper_blossom_end_rot_Bing_0077.jpg', 'bell_pepper_frogeye_leaf_spot_Bing_0008.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0049.jpg', 'bell_pepper_powdery_mildew_Bing_0035.jpg', 'bell_pepper_powdery_mildew_Google_0118.jpg', 'blueberry_anthracnose_Bing_0006.jpg', 'blueberry_anthracnose_Bing_0145.jpg', 'blueberry_botrytis_blight_Bing_0015.jpg', 'blueberry_botrytis_blight_Bing_0021.jpg', 'blueberry_mummy_berry_Bing_0024.jpg', 'blueberry_mummy_berry_Google_0038.jpg', 'blueberry_rust_google_0004.jpg', 'blueberry_rust_google_0053.jpg', 'blueberry_scorch_Bing_0011.jpg', 'blueberry_scorch_Bing_0031.jpg', 'blueberry_scorch_Bing_0062.jpg', 'broccoli_alternaria_leaf_spot_Bing_0000.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_downy_mildew_30.jpg', 'broccoli_downy_mildew_40.jpg', 'broccoli_ring_spot_Bing_0002.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_3.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0002.jpg', 'cabbage_black_rot_Bing_0018.jpg', 'cabbage_black_rot_Bing_0039.jpg', 'cabbage_black_rot_Bing_0080.jpg', 'cabbage_black_rot_Bing_0114.jpg', 'cabbage_downy_mildew_Bing_0012.jpg', 'cabbage_downy_mildew_Bing_0124.jpg', 'carrot_alternaria_leaf_blight_Bing_0046.jpg', 'carrot_alternaria_leaf_blight_Bing_0071.jpg', 'carrot_cavity_spot_google_0000.jpg', 'carrot_cavity_spot_google_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'carrot_cercospora_leaf_blight_Bing_0212.jpg', 'cauliflower_alternaria_leaf_spot_google_0003.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'cauliflower_bacterial_soft_rot_Bing_0205.jpg', 'celery_anthracnose_google_0007.jpg', 'celery_anthracnose_google_0035.jpg', 'celery_early_blight_52.jpg', 'celery_early_blight_google_0001.jpg', 'celery_early_blight_google_0009.jpg', 'cherry_leaf_spot_google_0180.jpg', 'cherry_leaf_spot_google_0216.jpg', 'cherry_powdery_mildew_107.jpg', 'cherry_powdery_mildew_2.jpg', 'citrus_canker_13.jpg', 'citrus_canker_147.jpg', 'citrus_canker_163.jpg', 'citrus_canker_247.jpg', 'citrus_canker_363.jpg', 'citrus_canker_76.jpg', 'citrus_canker_89.jpg', 'citrus_canker_google_0071.jpg', 'citrus_greening_disease_29.jpg', 'citrus_greening_disease_48.jpg', 'citrus_greening_disease_google_0120.jpg', 'citrus_greening_disease_google_0251.jpg', 'coffee_berry_blotch_Bing_0027.jpg', 'coffee_berry_blotch_Bing_0114.jpg', 'coffee_berry_blotch_Bing_0120.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_15.jpg', 'coffee_leaf_rust_google_0016.jpg', 'coffee_leaf_rust_google_0135.jpg', 'corn_gray_leaf_spot_31.jpg', 'corn_gray_leaf_spot_35.jpg', 'corn_gray_leaf_spot_google_0074.jpg', 'corn_northern_leaf_blight_101.jpg', 'corn_northern_leaf_blight_14.jpg', 'corn_northern_leaf_blight_81.jpg', 'corn_rust_80.jpg', 'corn_rust_google_0078.jpg', 'corn_smut_google_0013.jpg', 'corn_smut_google_0063.jpg', 'corn_smut_google_0079.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_156.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_angular_leaf_spot_41.jpg', 'cucumber_angular_leaf_spot_google_0043.jpg', 'cucumber_bacterial_wilt_109.jpg', 'cucumber_bacterial_wilt_313.jpg', 'cucumber_bacterial_wilt_50.jpg', 'cucumber_powdery_mildew_58.jpg', 'cucumber_powdery_mildew_70.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0159.jpg', 'eggplant_cercospora_leaf_spot_12.jpg', 'eggplant_cercospora_leaf_spot_33.jpg', 'eggplant_cercospora_leaf_spot_40.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0016.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0261.jpg', 'eggplant_phytophthora_blight_Bing_0000.jpg', 'eggplant_phytophthora_blight_Bing_0002.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_leaf_blight_google_0004.jpg', 'garlic_leaf_blight_google_0210.jpg', 'garlic_rust_159.jpg', 'garlic_rust_18.jpg', 'ginger_leaf_spot_11.jpg', 'ginger_leaf_spot_google_0001.jpg', 'ginger_sheath_blight_53.jpg', 'ginger_sheath_blight_89.jpg', 'grape_black_rot_12.jpg', 'grape_black_rot_216.jpg', 'grape_black_rot_google_0013.jpg', 'grape_downy_mildew_251.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_google_0032.jpg', 'grape_downy_mildew_google_0052.jpg', 'grape_leaf_spot_101.jpg', 'grape_leaf_spot_168.jpg', 'grapevine_leafroll_disease_google_0001.jpg', 'grapevine_leafroll_disease_google_0181.jpg', 'lettuce_downy_mildew_google_0015.jpg', 'lettuce_downy_mildew_google_0027.jpg', 'lettuce_mosaic_virus_16.jpg', 'lettuce_mosaic_virus_24.jpg', 'maple_tar_spot_google_0005.jpg', 'maple_tar_spot_google_0089.jpg', 'peach_anthracnose_Bing_0002.jpg', 'peach_anthracnose_Bing_0013.jpg', 'peach_brown_rot_Bing_0000.jpg', 'peach_brown_rot_Bing_0092.jpg', 'peach_brown_rot_Bing_0114.jpg', 'peach_leaf_curl_34.jpg', 'peach_leaf_curl_google_0020.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_leaf_curl_google_0074.jpg', 'peach_rust_Bing_0001.jpg', 'peach_scab_Bing_0007.jpg', 'peach_scab_Bing_0083.jpg', 'plum_bacterial_spot_Bing_0013.jpg', 'plum_bacterial_spot_Bing_0082.jpg', 'plum_brown_rot_Bing_0001.jpg', 'plum_brown_rot_Bing_0097.jpg', 'plum_pocket_disease_google_0028.jpg', 'plum_pocket_disease_google_0032.jpg', 'plum_pocket_disease_google_0117.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Bing_0054.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0024.jpg', 'plum_rust_Google_0036.jpg', 'potato_early_blight_6.jpg', 'potato_early_blight_google_0095.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_36.jpg', 'potato_late_blight_96.jpg', 'raspberry_fire_blight_Bing_0018.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_gray_mold_Bing_0000.jpg', 'raspberry_gray_mold_Bing_0001.jpg', 'raspberry_leaf_spot_Bing_0073.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0183.jpg', 'rice_blast_151.jpg', 'rice_blast_87.jpg', 'rice_sheath_blight_148.jpg', 'rice_sheath_blight_google_0027.jpg', 'soybean_bacterial_blight_Bing_0057.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Baidu_0010.jpg', 'soybean_brown_spot_Bing_0353.jpg', 'soybean_downy_mildew_Bing_0078.jpg', 'soybean_downy_mildew_Bing_0143.jpg', 'soybean_frog_eye_leaf_spot_Bing_0141.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_frog_eye_leaf_spot_Google_0221.jpg', 'soybean_mosaic_Baidu_0059.jpg', 'soybean_mosaic_Baidu_0089.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'soybean_rust_Bing_0075.jpg', 'soybean_rust_Bing_0096.jpg', 'squash_powdery_mildew_129.jpg', 'squash_powdery_mildew_23.jpg', 'squash_powdery_mildew_6.jpg', 'squash_powdery_mildew_google_0105.jpg', 'strawberry_anthracnose_5.jpg', 'strawberry_anthracnose_google_0001.jpg', 'strawberry_leaf_scorch_google_0037.jpg', 'strawberry_leaf_scorch_google_0133.jpg', 'tobacco_blue_mold_Bing_0024.jpg', 'tobacco_blue_mold_Bing_0096.jpg', 'tobacco_brown_spot_Bing_0001.jpg', 'tobacco_brown_spot_Bing_0047.jpg', 'tobacco_brown_spot_Bing_0055.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_frogeye_leaf_spot_Google_0052.jpg', 'tobacco_mosaic_virus_117.jpg', 'tobacco_mosaic_virus_4.jpg', 'tomato_bacterial_leaf_spot_120.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0177.jpg', 'tomato_late_blight_10.jpg', 'tomato_late_blight_182.jpg', 'tomato_late_blight_205.jpg', 'tomato_leaf_mold_10.jpg', 'tomato_leaf_mold_148.jpg', 'tomato_mosaic_virus_15.jpg', 'tomato_mosaic_virus_google_0049.jpg', 'tomato_septoria_leaf_spot_10.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_80.jpg', 'tomato_septoria_leaf_spot_google_0027.jpg', 'tomato_septoria_leaf_spot_google_0070.jpg', 'tomato_septoria_leaf_spot_google_0128.jpg', 'tomato_yellow_leaf_curl_virus_15.jpg', 'tomato_yellow_leaf_curl_virus_17.jpg', 'tomato_yellow_leaf_curl_virus_40.jpg', 'tomato_yellow_leaf_curl_virus_99.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0235.jpg', 'wheat_head_scab_Baidu_0481.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0288.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0058.jpg', 'wheat_leaf_rust_Google_0328.jpg', 'wheat_loose_smut_Bing_0109.jpg', 'wheat_loose_smut_Bing_0283.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0181.jpg', 'wheat_powdery_mildew_Bing_0014.jpg', 'wheat_powdery_mildew_Google_0014.jpg', 'wheat_septoria_blotch_Google_0115.jpg', 'wheat_septoria_blotch_google_septoria (18).jpg', 'wheat_stem_rust_Baidu_0001.jpg', 'wheat_stem_rust_Bing_0217.jpg', 'wheat_stem_rust_Google_0274.jpg', 'wheat_stripe_rust_Baidu_0143.jpg', 'wheat_stripe_rust_Bing_0007.jpg', 'wheat_stripe_rust_Bing_0257.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'zucchini_bacterial_wilt_Bing_0105.jpg', 'zucchini_bacterial_wilt_Bing_0188.jpg', 'zucchini_bacterial_wilt_Bing_0260.jpg', 'zucchini_bacterial_wilt_Google_0060.jpg', 'zucchini_downy_mildew_Bing_0035.jpg', 'zucchini_downy_mildew_Bing_0259.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0155.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_yellow_mosaic_virus_11.jpg', 'zucchini_yellow_mosaic_virus_google_0038.jpg', 'zucchini_yellow_mosaic_virus_google_0067.jpg']
print(f"Reusing the exact {len(NAMES_300)}-image stratified subset from Stage 4")

sub_df = test_df[test_df["Name"].isin(NAMES_300)].reset_index(drop=True)
assert len(sub_df) == len(NAMES_300), f"matched {len(sub_df)} / {len(NAMES_300)} -- check Name vs image column"

def build_classifier(num_classes):
    return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes)

clf = build_classifier(NUM_CLASSES)
clf = load_state(clf, CLS_CKPT).to(device).eval()

cls_tf = T.Compose([T.Resize((IMG_SIZE_CLS, IMG_SIZE_CLS)), T.ToTensor(), T.Normalize(MEAN, STD)])
cam_extractor = GradCAM(model=clf, target_layers=[clf.stages[-1]])

DEL_STEPS = 10

def cam_gt_metrics(cam, gt_mask):
    if gt_mask.sum() == 0: return {"iou": np.nan, "pointing_hit": np.nan}
    thresh = np.quantile(cam, 1 - gt_mask.mean())
    cam_bin = cam >= thresh
    inter = (cam_bin & gt_mask).sum(); union = (cam_bin | gt_mask).sum()
    iou = float(inter/union) if union > 0 else np.nan
    py, px = np.unravel_index(np.argmax(cam), cam.shape)
    return {"iou": iou, "pointing_hit": bool(gt_mask[py, px])}

def deletion_auc_cls(model, x, cam, pred_idx, steps=DEL_STEPS):
    H, W = x.shape[-2:]; order = np.argsort(-cam.ravel()); npix = order.size
    baseline = x.mean(dim=(2,3), keepdim=True); fracs = np.linspace(0, 1, steps+1); probs = []
    with torch.no_grad():
        for f in fracs:
            k = int(f*npix); xin = x.clone()
            if k > 0:
                mask = torch.zeros(npix, dtype=torch.bool, device=x.device)
                mask[order[:k]] = True; mask = mask.reshape(H, W)
                for c in range(xin.shape[1]): xin[0,c][mask] = baseline[0,c,0,0]
            p = torch.softmax(model(xin), 1)[0, pred_idx].item()
            probs.append(p)
    return float(np.trapz(probs, fracs))

rows = []
for r in sub_df.itertuples():
    img = Image.open(r.filepath).convert("RGB")
    gt = np.array(Image.open(r.maskpath))
    gt_bin = (gt > 0)
    x = cls_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_idx = int(clf(x).argmax(1))
    cam = cam_extractor(input_tensor=x, targets=None)[0]
    cam_rs = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(gt_bin.shape[::-1])) / 255.0
    m = cam_gt_metrics(cam_rs, gt_bin)
    delauc = deletion_auc_cls(clf, x, cam_rs if cam_rs.shape == x.shape[-2:] else
                               np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE_CLS, IMG_SIZE_CLS)))/255.0,
                               pred_idx)
    rows.append({"image": r.Name, "disease": r.Disease, "cam_gt_iou": m["iou"],
                 "pointing_hit": m["pointing_hit"], "deletion_auc": delauc})

clf_faith = pd.DataFrame(rows)
clf_faith.to_csv(OUTDIR/"classifier_gradcam_vs_gt.csv", index=False)
print(clf_faith[["cam_gt_iou","pointing_hit","deletion_auc"]].mean())
print(f"\nBootstrap 95% CIs (n_boot=2000):")
rng = np.random.RandomState(SEED)
summary1 = {"n_images": int(len(clf_faith))}
for col in ["cam_gt_iou","pointing_hit","deletion_auc"]:
    vals = clf_faith[col].dropna().values
    boot = [rng.choice(vals, len(vals), replace=True).mean() for _ in range(2000)]
    lo, hi = float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5))
    summary1[col] = {"mean": float(vals.mean()), "ci95_lo": lo, "ci95_hi": hi}
    print(f"  {col}: mean={vals.mean():.4f}  95% CI=[{lo:.4f}, {hi:.4f}]")
json.dump(summary1, open(OUTDIR/"classifier_gradcam_vs_gt_SUMMARY.json", "w"), indent=2)
print(f"\nSaved: {OUTDIR/'classifier_gradcam_vs_gt.csv'} (per-image) and _SUMMARY.json (means + CIs, for direct use in the response letter/paper)")
print("Compare these numbers directly to Table 10's five CNN segmenters (mean CAM-GT IoU ~0.43,")
print("pointing-hit ~0.88) to see whether the classifier's own Grad-CAM is similarly faithful.")

Reusing the exact 300-image stratified subset from Stage 4
cam_gt_iou      0.267061
pointing_hit    0.523333
deletion_auc    0.152250
dtype: float64

Bootstrap 95% CIs (n_boot=2000):
  cam_gt_iou: mean=0.2671  95% CI=[0.2414, 0.2906]
  pointing_hit: mean=0.5233  95% CI=[0.4633, 0.5800]
  deletion_auc: mean=0.1523  95% CI=[0.1349, 0.1703]

Saved: /kaggle/working/stage6_results/classifier_gradcam_vs_gt.csv (per-image) and _SUMMARY.json (means + CIs, for direct use in the response letter/paper)
Compare these numbers directly to Table 10's five CNN segmenters (mean CAM-GT IoU ~0.43,
pointing-hit ~0.88) to see whether the classifier's own Grad-CAM is similarly faithful.


## Section 2 — Validation-set-based re-ranking (no test-set peeking)

The Stage 1/Stage 2 aggregators picked "the best classifier" and "the best segmenter" using
**test-set** metrics. This re-runs every already-trained checkpoint on the **validation** set
and re-ranks there; whichever model wins is the one whose *test-set* number should be reported
as the confirmatory result.

In [4]:
# ============ SECTION 2a: CLASSIFIERS -- VALIDATION-SET RE-RANKING ============
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

print("timm version:", timm.__version__)

def build_backbone(name, num_classes, global_pool=None):
    import torchvision
    gp_kwargs = {"global_pool": global_pool} if global_pool is not None else {}
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ViTBase16":
        try:
            m = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes, **gp_kwargs)
        except TypeError:
            m = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes)
        return m, IMG_SIZE_CLS_TF
    if name == "SwinTiny":
        try:
            m = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes, **gp_kwargs)
        except TypeError:
            m = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes)
        return m, IMG_SIZE_CLS_TF
    if name == "GatedFusionLight":
        import torchvision
        class _Gated(nn.Module):
            def __init__(self, ba, da, bb, db, nc, gh=128, hh=256, dr=0.3):
                super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
                self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
                self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
            def forward(self, x):
                fa, fb = self.branch_a(x), self.branch_b(x)
                g = self.gate(torch.cat([fa,fb], 1))
                return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))
        mb = torchvision.models.mobilenet_v3_large(weights=None)
        da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
        b = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0, global_pool="avg")
        return _Gated(mb, da, b, b.num_features, num_classes), IMG_SIZE_CLS_TF  # GatedFusionLight trained at 224, not 384 (confirmed from Stage 1 train_one() source: img_size hardcoded to 224 for FUSION_MODELS)
    raise ValueError(name)

def evaluate_classifier_on(df, model, img_size):
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for r in df.itertuples():
            x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            p = int(model(x).argmax(1))
            y_true.append(label2idx[r.label]); y_pred.append(p)
    return (accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro", zero_division=0),
            matthews_corrcoef(y_true, y_pred))

# --- diagnostic auto-fix for the two transformer classifiers only ---
# The first run of this cell (2026-09-19) gave near-chance val accuracy for ViTBase16 (9.2%) and
# SwinTiny (0.9%) despite both scoring ~76% on the original TEST-set evaluation with the SAME
# checkpoint, SAME 224x224 preprocessing, and a CLEAN state_dict load (zero missing/unexpected
# keys) -- which rules out a path/resolution/label bug. The most likely cause is timm-version
# drift: the pip-install cell has no version pin, so this Kaggle run may have installed a
# different timm release than whichever was live when Stage 1 originally trained these models,
# and timm has changed default global-pooling/head behaviour for ViT/Swin across releases before
# -- a change that leaves the state_dict's keys/shapes identical (so the load looks clean) while
# silently feeding the trained classifier head features it was never trained on.
# Fix strategy: rather than guess the right config, empirically try a short list of plausible
# global_pool overrides on a QUICK slice of val_df and keep whichever config actually recovers
# sane (well-above-chance) accuracy, then use that config for the full val evaluation below.
# If NONE of the candidates recover sane accuracy, this is flagged explicitly rather than
# silently accepted -- in that case the fix is to pin timm to the exact version used when
# Stage 1 trained (check that session's pip freeze / Kaggle kernel log if available).

QUICK_N = 60
_stride = max(1, len(val_df) // QUICK_N)
quick_df = val_df.iloc[::_stride][:QUICK_N]
CHANCE = 1.0 / NUM_CLASSES

def quick_acc(name, global_pool):
    model, img_size = build_backbone(name, NUM_CLASSES, global_pool=global_pool)
    model = load_state(model, CLS_CKPTS[name]).to(device)
    acc, _, _ = evaluate_classifier_on(quick_df, model, img_size)
    del model; torch.cuda.empty_cache()
    return acc

TF_POOL_CANDIDATES = [None, "avg", "token", "map"]
best_pool = {}
for tf_name in ["ViTBase16", "SwinTiny"]:
    if not CLS_CKPTS[tf_name].exists():
        print(f"  [skip diag] {tf_name}: checkpoint not found"); continue
    print(f"\nDiagnosing {tf_name} on a quick {len(quick_df)}-image slice (chance level ~= {CHANCE:.4f}):")
    results = {}
    for gp in TF_POOL_CANDIDATES:
        try:
            a = quick_acc(tf_name, gp)
            results[gp] = a
            print(f"    global_pool={gp!r}: quick_acc={a:.4f}")
        except Exception as e:
            print(f"    global_pool={gp!r}: FAILED ({type(e).__name__}: {e})")
    if results:
        winner = max(results, key=results.get)
        recovered = results[winner] > 2 * CHANCE
        best_pool[tf_name] = winner if recovered else None
        flag = "OK, clearly above chance -- using this config" if recovered else \
               "STILL AT CHANCE -- global_pool was not the cause; falling back to default and flagging for manual investigation"
        print(f"  -> best candidate for {tf_name}: global_pool={winner!r} (quick_acc={results[winner]:.4f}) [{flag}]")
    else:
        best_pool[tf_name] = None
        print(f"  -> all candidates failed to even run for {tf_name}; using default config (unresolved)")

val_ranking = []
still_broken = []
for name, ckpt in CLS_CKPTS.items():
    if not ckpt.exists():
        print(f"  [skip] {name}: checkpoint not found at {ckpt}"); continue
    gp = best_pool.get(name, None)
    model, img_size = build_backbone(name, NUM_CLASSES, global_pool=gp)
    model = load_state(model, ckpt).to(device)
    acc, f1m, mcc = evaluate_classifier_on(val_df, model, img_size)
    val_ranking.append({"model": name, "val_accuracy": acc, "val_f1_macro": f1m, "val_mcc": mcc,
                         "global_pool_used": gp if gp is not None else "default"})
    tag = f"  [global_pool={gp!r}]" if gp else ""
    print(f"  {name}: val_acc={acc:.4f} val_f1_macro={f1m:.4f}{tag}")
    if name in ("ViTBase16", "SwinTiny") and acc < 2 * CHANCE:
        still_broken.append(name)
    del model; torch.cuda.empty_cache()

val_rank_df = pd.DataFrame(val_ranking).sort_values("val_f1_macro", ascending=False)
val_rank_df.to_csv(OUTDIR/"classifier_val_reranking.csv", index=False)
print("\nValidation-set ranking (this is what model selection SHOULD have used):")
print(val_rank_df.to_string(index=False))
print(f"\nWinner by val_f1_macro: {val_rank_df.iloc[0]['model']}  "
      f"(paper currently reports ConvNeXtV2-Tiny as best by TEST f1_macro -- confirm these agree)")

if still_broken:
    print(f"\n*** WARNING: {still_broken} still at/near chance level after the global_pool auto-fix. ***")
    print("*** Do NOT use this table for those model(s) until fixed -- try pinning timm to the exact ***")
    print("*** version used when Stage 1 trained (check that Kaggle kernel's pip freeze / run log). ***")
else:
    print("\nAll 8 classifiers are now well above chance level -- table is ready for the response letter.")


timm version: 1.0.26

Diagnosing ViTBase16 on a quick 60-image slice (chance level ~= 0.0087):
    global_pool=None: quick_acc=0.0833
  [load_state:ViTBase16_best.pt] missing=2 unexpected=2
    global_pool='avg': quick_acc=0.0000
    global_pool='token': quick_acc=0.0833
  [load_state:ViTBase16_best.pt] missing=13 unexpected=0
    global_pool='map': quick_acc=0.0333
  -> best candidate for ViTBase16: global_pool=None (quick_acc=0.0833) [OK, clearly above chance -- using this config]

Diagnosing SwinTiny on a quick 60-image slice (chance level ~= 0.0087):
    global_pool=None: quick_acc=0.0333
    global_pool='avg': quick_acc=0.0333
    global_pool='token': FAILED (AssertionError: )
    global_pool='map': FAILED (AssertionError: )
  -> best candidate for SwinTiny: global_pool=None (quick_acc=0.0333) [OK, clearly above chance -- using this config]
  MobileNetV3Large: val_acc=0.6749 val_f1_macro=0.5917
  EfficientNetB0: val_acc=0.6395 val_f1_macro=0.5537
  ResNet50: val_acc=0.6749 val_f1_

In [5]:
# ============ SECTION 2b: SEGMENTERS -- VALIDATION-SET RE-RANKING ============
from transformers import SegformerForSemanticSegmentation

SMP_SPECS = {
    "UNet_resnet34":                 (smp.Unet,          "resnet34"),
    "UNetPP_resnet50":               (smp.UnetPlusPlus,  "resnet50"),
    "DeepLabV3Plus_resnet50":        (smp.DeepLabV3Plus, "resnet50"),
    "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
    "FPN_resnet50":                  (smp.FPN,           "resnet50"),
}
SEGFORMER_HF = {"SegFormer_b2": "nvidia/mit-b2"}

class SegNet(nn.Module):
    def __init__(self, name, nc):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]
            self.net = arch(encoder_name=enc, encoder_weights=None, in_channels=3, classes=nc); self.kind = "smp"
        else:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF[name], num_labels=nc, ignore_mismatched_sizes=True); self.kind = "hf"
    def forward(self, x):
        if self.kind == "smp": return self.net(x)
        lg = self.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

seg_eval_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])
mask_tf = T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG), interpolation=T.InterpolationMode.NEAREST)

def evaluate_segmenter_on(df, model):
    model.eval(); ious = []
    with torch.no_grad():
        for r in df.itertuples():
            x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
            pr = (model(x).argmax(1)[0].cpu().numpy() > 0)
            inter = (gt & pr).sum(); union = (gt | pr).sum()
            ious.append(inter/union if union > 0 else 1.0)
    return float(np.mean(ious))

seg_val_ranking = []
for name, ckpt in SEG_CKPTS.items():
    if not ckpt.exists():
        print(f"  [skip] {name}: checkpoint not found at {ckpt}"); continue
    model = SegNet(name, 2)
    model = load_state(model, ckpt).to(device)
    fg_iou = evaluate_segmenter_on(val_df, model)
    seg_val_ranking.append({"model": name, "val_fg_iou_macro": fg_iou})
    print(f"  {name}: val_fg_iou (per-image macro) = {fg_iou:.4f}")
    del model; torch.cuda.empty_cache()

seg_val_df = pd.DataFrame(seg_val_ranking).sort_values("val_fg_iou_macro", ascending=False)
seg_val_df.to_csv(OUTDIR/"segmenter_val_reranking.csv", index=False)
print("\nValidation-set segmenter ranking:")
print(seg_val_df.to_string(index=False))

  UNet_resnet34: val_fg_iou (per-image macro) = 0.5439
  UNetPP_resnet50: val_fg_iou (per-image macro) = 0.5258
  FPN_resnet50: val_fg_iou (per-image macro) = 0.5445
  DeepLabV3Plus_resnet50: val_fg_iou (per-image macro) = 0.5255
  DeepLabV3Plus_efficientnet-b3: val_fg_iou (per-image macro) = 0.5604


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

  SegFormer_b2: val_fg_iou (per-image macro) = 0.5265

Validation-set segmenter ranking:
                        model  val_fg_iou_macro
DeepLabV3Plus_efficientnet-b3          0.560424
                 FPN_resnet50          0.544493
                UNet_resnet34          0.543911
                 SegFormer_b2          0.526533
              UNetPP_resnet50          0.525816
       DeepLabV3Plus_resnet50          0.525527


## Section 3 — Deployed pipeline on "clean" vs. cross-split-URL-leaked test images

250 of the 1,561 test images (16%) share their source `URL` with an image in train/val --
likely the same acquisition/photo session. This checks whether the deployed classifier +
segmenter perform differently on those 250 vs. the 1,311 "clean" images.

In [6]:
# ============ SECTION 3: CLEAN VS URL-LEAKED TEST SUBSET ============
LEAKED_NAMES = set(['apple_black_rot_144.jpg', 'apple_mosaic_virus_43.jpg', 'apple_mosaic_virus_57.jpg', 'apple_rust_74.jpg', 'apple_rust_96.jpg', 'apple_scab_50.jpg', 'apple_scab_8.jpg', 'banana_black_leaf_streak_Baidu_0006.jpg', 'banana_black_leaf_streak_banana black sigatoka (42).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_cigar_end_rot_Bing_0002.jpg', 'banana_cigar_end_rot_Bing_0014.jpg', 'banana_panama_disease_6.jpg', 'bean_rust_178.jpg', 'bean_rust_189.jpg', 'bean_rust_google_0130.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0005.jpg', 'bell_pepper_blossom_end_rot_Bing_0008.jpg', 'bell_pepper_blossom_end_rot_Bing_0046.jpg', 'bell_pepper_blossom_end_rot_Bing_0112.jpg', 'bell_pepper_blossom_end_rot_Google_0044.jpg', 'bell_pepper_powdery_mildew_Bing_0001.jpg', 'blueberry_scorch_Bing_0011.jpg', 'broccoli_alternaria_leaf_spot_Bing_0019.jpg', 'broccoli_alternaria_leaf_spot_Bing_0021.jpg', 'broccoli_alternaria_leaf_spot_Bing_0045.jpg', 'broccoli_alternaria_leaf_spot_Bing_0060.jpg', 'broccoli_alternaria_leaf_spot_Google_0005.jpg', 'broccoli_downy_mildew_9.jpg', 'cabbage_alternaria_leaf_spot_48.jpg', 'cabbage_alternaria_leaf_spot_google_0026.jpg', 'cabbage_alternaria_leaf_spot_google_0148.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0007.jpg', 'cabbage_black_rot_Bing_0218 - Copy.jpg', 'cabbage_black_rot_Google_0101.jpg', 'cabbage_downy_mildew_Bing_0039.jpg', 'carrot_alternaria_leaf_blight_Bing_0075.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'cauliflower_alternaria_leaf_spot_google_0028.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'celery_anthracnose_google_0035.jpg', 'citrus_canker_107.jpg', 'citrus_canker_11.jpg', 'citrus_canker_130.jpg', 'citrus_canker_162.jpg', 'citrus_canker_163.jpg', 'citrus_canker_197.jpg', 'citrus_canker_251.jpg', 'citrus_canker_256.jpg', 'citrus_canker_276.jpg', 'citrus_canker_340.jpg', 'citrus_canker_388.jpg', 'citrus_canker_4.jpg', 'citrus_canker_50.jpg', 'citrus_canker_60.jpg', 'citrus_canker_76.jpg', 'coffee_berry_blotch_Bing_0006.jpg', 'coffee_berry_blotch_Bing_0010.jpg', 'coffee_berry_blotch_Bing_0037.jpg', 'coffee_berry_blotch_Bing_0049.jpg', 'coffee_berry_blotch_Bing_0099.jpg', 'coffee_berry_blotch_Bing_0117.jpg', 'coffee_berry_blotch_Bing_0140.jpg', 'coffee_brown_eye_spot_Bing_0006.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'corn_northern_leaf_blight_2.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_141.jpg', 'cucumber_angular_leaf_spot_261.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_bacterial_wilt_184.jpg', 'cucumber_bacterial_wilt_58.jpg', 'cucumber_powdery_mildew_10.jpg', 'cucumber_powdery_mildew_125.jpg', 'cucumber_powdery_mildew_178.jpg', 'cucumber_powdery_mildew_183.jpg', 'cucumber_powdery_mildew_198.jpg', 'cucumber_powdery_mildew_227.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0001.jpg', 'cucumber_powdery_mildew_google_0026.jpg', 'cucumber_powdery_mildew_google_0080.jpg', 'eggplant_cercospora_leaf_spot_96.jpg', 'eggplant_cercospora_leaf_spot_google_0030.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'garlic_leaf_blight_35.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_rust_56.jpg', 'garlic_rust_83.jpg', 'ginger_leaf_spot_161.jpg', 'ginger_sheath_blight_30.jpg', 'ginger_sheath_blight_89.jpg', 'ginger_sheath_blight_google_0007.jpg', 'grape_downy_mildew_115.jpg', 'grape_downy_mildew_202.jpg', 'grape_downy_mildew_223.jpg', 'grape_downy_mildew_241.jpg', 'grape_downy_mildew_29.jpg', 'grape_downy_mildew_49.jpg', 'grape_downy_mildew_82.jpg', 'lettuce_mosaic_virus_8.jpg', 'maple_tar_spot_google_0200.jpg', 'peach_brown_rot_Bing_0009.jpg', 'peach_brown_rot_Bing_0040.jpg', 'peach_brown_rot_Bing_0141.jpg', 'peach_brown_rot_Bing_0269.jpg', 'peach_leaf_curl_2.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_scab_Bing_0046.jpg', 'peach_scab_Bing_0083.jpg', 'peach_scab_Bing_0223.jpg', 'plum_bacterial_spot_Google_0057.jpg', 'plum_brown_rot_Bing_0000.jpg', 'plum_brown_rot_Bing_0079.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Google_0040.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_95.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0101.jpg', 'rice_sheath_blight_43.jpg', 'rice_sheath_blight_52.jpg', 'rice_sheath_blight_58.jpg', 'rice_sheath_blight_google_0000.jpg', 'soybean_bacterial_blight_Bing_0055.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_brown_spot_Bing_0025.jpg', 'soybean_brown_spot_Bing_0090.jpg', 'soybean_downy_mildew_Bing_0038.jpg', 'soybean_downy_mildew_Bing_0045.jpg', 'soybean_downy_mildew_Google_0061.jpg', 'soybean_frog_eye_leaf_spot_Baidu_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0004.jpg', 'soybean_frog_eye_leaf_spot_Bing_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0047.jpg', 'soybean_frog_eye_leaf_spot_Bing_0089.jpg', 'soybean_frog_eye_leaf_spot_Bing_0095.jpg', 'soybean_frog_eye_leaf_spot_Bing_0155.jpg', 'soybean_frog_eye_leaf_spot_Google_0048.jpg', 'soybean_frog_eye_leaf_spot_Google_0056.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_mosaic_Bing_0062.jpg', 'soybean_rust_Bing_0032.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'squash_powdery_mildew_114.jpg', 'squash_powdery_mildew_google_0006.jpg', 'squash_powdery_mildew_google_0079.jpg', 'squash_powdery_mildew_google_0102.jpg', 'squash_powdery_mildew_google_0191.jpg', 'tobacco_blue_mold_Bing_0107.jpg', 'tobacco_blue_mold_Bing_0127.jpg', 'tobacco_blue_mold_Google_0008.jpg', 'tobacco_brown_spot_Bing_0003.jpg', 'tobacco_brown_spot_Bing_0044.jpg', 'tobacco_brown_spot_Bing_0159.jpg', 'tobacco_brown_spot_Google_0029.jpg', 'tobacco_frogeye_leaf_spot_Bing_0000.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_mosaic_virus_161.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_bacterial_leaf_spot_31.jpg', 'tomato_bacterial_leaf_spot_78.jpg', 'tomato_bacterial_leaf_spot_google_0087.jpg', 'tomato_bacterial_leaf_spot_google_0163.jpg', 'tomato_early_blight_10.jpg', 'tomato_early_blight_136.jpg', 'tomato_early_blight_159.jpg', 'tomato_early_blight_4.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0041.jpg', 'tomato_early_blight_google_0267.jpg', 'tomato_late_blight_199.jpg', 'tomato_leaf_mold_182.jpg', 'tomato_leaf_mold_259.jpg', 'tomato_leaf_mold_59.jpg', 'tomato_mosaic_virus_55.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_63.jpg', 'tomato_septoria_leaf_spot_google_0205.jpg', 'tomato_yellow_leaf_curl_virus_4.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0202.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0217.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0275.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Google_0141.jpg', 'wheat_head_scab_Baidu_0065.jpg', 'wheat_head_scab_Baidu_0310.jpg', 'wheat_head_scab_Baidu_0368.jpg', 'wheat_head_scab_Baidu_0404.jpg', 'wheat_head_scab_Bing_0023.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0111.jpg', 'wheat_head_scab_Bing_0206.jpg', 'wheat_head_scab_Bing_0219.jpg', 'wheat_head_scab_Google_0083.jpg', 'wheat_head_scab_Google_0309.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0374.jpg', 'wheat_loose_smut_Baidu_0036.jpg', 'wheat_loose_smut_Bing_0008.jpg', 'wheat_loose_smut_Bing_0097.jpg', 'wheat_loose_smut_Google_0065.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0040.jpg', 'wheat_powdery_mildew_Baidu_0082.jpg', 'wheat_powdery_mildew_Baidu_0083.jpg', 'wheat_powdery_mildew_Bing_0023.jpg', 'wheat_powdery_mildew_Bing_0163.jpg', 'wheat_powdery_mildew_Google_0101.jpg', 'wheat_septoria_blotch_Baidu_0098.jpg', 'wheat_septoria_blotch_Bing_0045.jpg', 'wheat_septoria_blotch_Google_0016.jpg', 'wheat_septoria_blotch_Google_0058.jpg', 'wheat_septoria_blotch_Google_0174.jpg', 'wheat_septoria_blotch_Google_0206.jpg', 'wheat_septoria_blotch_Google_0309.jpg', 'wheat_septoria_blotch_blotch (22).jpg', 'wheat_septoria_blotch_blotch (69).jpg', 'wheat_septoria_blotch_blotch (79).jpg', 'wheat_stem_rust_Bing_0002.jpg', 'wheat_stem_rust_Bing_0016.jpg', 'wheat_stem_rust_Bing_0040.jpg', 'wheat_stem_rust_Bing_0070.jpg', 'wheat_stem_rust_Bing_0206.jpg', 'wheat_stem_rust_Bing_0247.jpg', 'wheat_stem_rust_Google_0071.jpg', 'wheat_stripe_rust_Baidu_0349.jpg', 'wheat_stripe_rust_Baidu_0401.jpg', 'wheat_stripe_rust_Baidu_0484.jpg', 'wheat_stripe_rust_Bing_0137.jpg', 'wheat_stripe_rust_Bing_0173.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'wheat_stripe_rust_Google_0223.jpg', 'zucchini_bacterial_wilt_Bing_0019.jpg', 'zucchini_powdery_mildew_Bing_0002.jpg', 'zucchini_powdery_mildew_Bing_0004.jpg', 'zucchini_powdery_mildew_Bing_0052.jpg', 'zucchini_powdery_mildew_Bing_0054.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0289.jpg', 'zucchini_powdery_mildew_Bing_0336.jpg', 'zucchini_powdery_mildew_Bing_0337.jpg', 'zucchini_powdery_mildew_Google_0045.jpg'])
print(f"Leaked-URL test images: {len(LEAKED_NAMES)} / {len(test_df)}")

clf2 = build_classifier(NUM_CLASSES)
clf2 = load_state(clf2, CLS_CKPT).to(device).eval()
seg2 = SegNet("DeepLabV3Plus_efficientnet-b3", 2)
seg2 = load_state(seg2, SEG_CKPT_BEST).to(device).eval()

rows3 = []
with torch.no_grad():
    for r in test_df.itertuples():
        x_cls = T.Compose([T.Resize((IMG_SIZE_CLS,IMG_SIZE_CLS)), T.ToTensor(), T.Normalize(MEAN,STD)])(
            Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
        pred = CLASS_NAMES[int(clf2(x_cls).argmax(1))]
        rows3.append({"image": r.Name, "leaked": r.Name in LEAKED_NAMES,
                      "correct": pred == r.Disease})

leak_df = pd.DataFrame(rows3)
grp = leak_df.groupby("leaked")["correct"].agg(["mean","count"])
print(grp)
from scipy.stats import fisher_exact
ct = pd.crosstab(leak_df.leaked, leak_df.correct)
odds, p = fisher_exact(ct.values)
print(f"\nFisher's exact (leaked vs clean accuracy): odds={odds:.3f}, p={p:.4f}")
leak_df.to_csv(OUTDIR/"clean_vs_leaked_subset.csv", index=False)
summary3 = {
    "n_total": int(len(leak_df)), "n_leaked": int(leak_df.leaked.sum()), "n_clean": int((~leak_df.leaked).sum()),
    "accuracy_leaked": float(grp.loc[True, "mean"]) if True in grp.index else None,
    "accuracy_clean": float(grp.loc[False, "mean"]) if False in grp.index else None,
    "fisher_odds_ratio": float(odds), "fisher_p_value": float(p),
}
json.dump(summary3, open(OUTDIR/"clean_vs_leaked_SUMMARY.json", "w"), indent=2)
print(f"Saved: {OUTDIR/'clean_vs_leaked_subset.csv'} (per-image) and _SUMMARY.json (group accuracies + Fisher's exact, for direct use in the response letter/paper)")

Leaked-URL test images: 250 / 1561
            mean  count
leaked                 
False   0.676583   1311
True    0.676000    250

Fisher's exact (leaked vs clean accuracy): odds=0.997, p=1.0000
Saved: /kaggle/working/stage6_results/clean_vs_leaked_subset.csv (per-image) and _SUMMARY.json (group accuracies + Fisher's exact, for direct use in the response letter/paper)


## Section 4 — Ground-truth-independent CAM threshold sensitivity

Table 10's CAM–GT IoU thresholds the attribution map at the *ground-truth* lesion area
fraction (oracle-assisted, per Reviewers 3/4/5). This recomputes CAM–GT IoU on the same
300-image subset for all 6 segmenters using two GT-independent thresholds instead: a fixed
50th-percentile cut and Otsu's method, to check whether the CNN-vs-SegFormer gap survives.

In [7]:
# ============ SECTION 4: GT-INDEPENDENT THRESHOLD SENSITIVITY ============
# Addresses the Limitations-section overclaim flagged during re-analysis: Table 8/10's
# CAM-GT IoU uses a GT-AREA-DEPENDENT threshold (quantile matched to the true lesion-area
# fraction), so it is not a fair "faithfulness" number on its own -- it needs a GT-independent
# check alongside it. This reuses Stage 4's exact attribution code (gradcam_map for the 5 CNN
# segmenters via the SegNet class from Section 2b, attention-rollout for SegFormer-B2) on the
# same 300-image stratified subset (sub_df, from Section 1), and reports IoU under two
# thresholds that do NOT see the ground truth at all: the map's own median, and Otsu's method.

try:
    from skimage.filters import threshold_otsu
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])
    from skimage.filters import threshold_otsu

class _GradCAMHook:
    def __init__(self, module):
        self.activations = None; self.gradients = None
        self.fh = module.register_forward_hook(self._fwd)
        self.bh = module.register_full_backward_hook(self._bwd)
    def _fwd(self, m, inp, out): self.activations = out
    def _bwd(self, m, gi, go):  self.gradients  = go[0]
    def remove(self): self.fh.remove(); self.bh.remove()

def gradcam_map(model, x, nc):
    hook = _GradCAMHook(model.net.decoder)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits[:,1] if nc==2 else logits[:,1:].sum(1)
    score.sum().backward()
    acts  = hook.activations[0].detach()
    grads = hook.gradients[0].detach()
    hook.remove()
    w = grads.mean(dim=(1,2))
    cam = torch.relu((w[:,None,None]*acts).sum(0))
    cam = cam/(cam.max()+1e-8)
    cam = F.interpolate(cam[None,None], size=x.shape[-2:], mode="bilinear",
                        align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def segformer_attention_map(model, x):
    with torch.no_grad():
        out = model.net.segformer(pixel_values=x, output_attentions=True) \
              if hasattr(model.net,"segformer") else model.net(pixel_values=x, output_attentions=True)
    attns = out.attentions
    stages=[]; cur=[]; cur_n=None
    for a in attns:
        n=a.shape[-2]
        if cur_n is None or n==cur_n: cur.append(a); cur_n=n
        else: stages.append(cur); cur=[a]; cur_n=n
    stages.append(cur)
    rollout=None
    for a in stages[-1]:
        A = a[0].mean(0)
        A = A/A.sum(-1,keepdim=True).clamp_min(1e-8)
        A_hat = 0.5*A + 0.5*torch.eye(A.shape[0], device=A.device)
        rollout = A_hat if rollout is None else A_hat@rollout
    imp = rollout.mean(0)
    side = int(round(imp.shape[0]**0.5))
    grid = imp[:side*side].reshape(side,side).float()
    grid = grid/(grid.max()+1e-8)
    cam  = F.interpolate(grid[None,None], size=x.shape[-2:], mode="bilinear",
                         align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def explain(model, x, nc, is_transformer):
    return segformer_attention_map(model,x) if is_transformer else gradcam_map(model,x,nc)

def _iou(bin_map, gt_mask):
    inter = (bin_map & gt_mask).sum(); union = (bin_map | gt_mask).sum()
    return float(inter/union) if union > 0 else np.nan

def gt_independent_thresholds(cam, gt_mask):
    out = {}
    med = np.median(cam)
    out["fixed_p50"] = _iou(cam >= med, gt_mask)
    try:
        t = threshold_otsu(cam)
        out["otsu"] = _iou(cam >= t, gt_mask)
    except Exception:
        out["otsu"] = np.nan
    return out

rows4 = []
for name, ckpt in SEG_CKPTS.items():
    if not ckpt.exists():
        print(f"  [skip] {name}: checkpoint not found at {ckpt}"); continue
    model = SegNet(name, 2)
    model = load_state(model, ckpt).to(device).eval()
    is_tf = (name == "SegFormer_b2")
    for r in sub_df.itertuples():
        x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
        gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
        cam = explain(model, x, 2, is_tf)
        sens = gt_independent_thresholds(cam, gt)
        rows4.append({"model": name, "image": r.Name, "fixed_p50_iou": sens["fixed_p50"],
                       "otsu_iou": sens["otsu"]})
    del model; torch.cuda.empty_cache()
    print(f"  done: {name}")

sens_df = pd.DataFrame(rows4)
sens_df.to_csv(OUTDIR/"threshold_sensitivity.csv", index=False)
sens_summary = sens_df.groupby("model")[["fixed_p50_iou","otsu_iou"]].mean().round(4)
sens_summary.to_csv(OUTDIR/"threshold_sensitivity_SUMMARY.csv")
print("\nGT-independent threshold sensitivity (mean IoU per model):")
print(sens_summary.to_string())
print("\nCompare these to Table 8/10's GT-area-quantile-threshold IoU to show the reported")
print("numbers are not an artifact of that specific (GT-dependent) thresholding choice.")
print(f"\nSaved: {OUTDIR/'threshold_sensitivity.csv'} (per-image) and _SUMMARY.csv (per-model means, for direct use in the response letter/paper)")


  done: UNet_resnet34
  done: UNetPP_resnet50
  done: FPN_resnet50
  done: DeepLabV3Plus_resnet50
  done: DeepLabV3Plus_efficientnet-b3


Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

  done: SegFormer_b2

GT-independent threshold sensitivity (mean IoU per model):
                               fixed_p50_iou  otsu_iou
model                                                 
DeepLabV3Plus_efficientnet-b3         0.2164    0.4762
DeepLabV3Plus_resnet50                0.2192    0.4709
FPN_resnet50                          0.2112    0.4001
SegFormer_b2                          0.0624    0.0034
UNetPP_resnet50                       0.2077    0.4159
UNet_resnet34                         0.2139    0.4768

Compare these to Table 8/10's GT-area-quantile-threshold IoU to show the reported
numbers are not an artifact of that specific (GT-dependent) thresholding choice.

Saved: /kaggle/working/stage6_results/threshold_sensitivity.csv (per-image) and _SUMMARY.csv (per-model means, for direct use in the response letter/paper)


## Wrap-up

Download `stage6_results/*.csv` (Kaggle output) and hand them back for the response letter and
manuscript updates. Section 1 is the single highest-priority result -- it directly answers
Reviewers 4 and 5's most serious methodological objection about the deployed confidence flag.